## meta-llama/Llama-3.2-1B-Instruct

In [1]:
import evaluate
import json
import os
import pandas as pd
import torch

from datasets import Dataset
from pathlib import Path
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

W0620 09:23:29.079000 2212 venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0620 09:23:29.155000 2212 venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


### DIRECTORIES

In [2]:
PROJECT_PATH = Path(os.getcwd()).parent.parent
DS_PATH = PROJECT_PATH / "datasets" / "fine-tuning" / "conversational"
TRAINING_DS_PATH = DS_PATH / "ordered_training_ds_juliet.jsonl"
TESTING_DS_PATH = DS_PATH / "ordered_testing_ds_juliet.jsonl"

LOGS_PATH = PROJECT_PATH / "logs" / "meta-llama-3_2-1B-instruct"
OUTPUT_METRICS_PATH = PROJECT_PATH / "output_metrics"
OUTPUT_METRICS_FILE = OUTPUT_METRICS_PATH / "meta-llama-3_2-1B-instruct.txt"

MODEL_OUTPUT_PATH = PROJECT_PATH / "fine-tuned-models" / "meta-llama-3_2-1B-instruct"

LOGS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_METRICS_PATH.mkdir(exist_ok=True)
MODEL_OUTPUT_PATH.mkdir(exist_ok=True)

### PARAMETERS

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

### TRAINING AND TESTING DATASET

In [4]:
train_data = []
with open(TRAINING_DS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line.strip()))

first_15K_training_examples = train_data[:15000]

### Refine dataset to Input - Label

In [5]:
training_dataframe = pd.DataFrame(first_15K_training_examples, columns=["messages", "cwe", "code", "filename", "type"])[["messages"]]

### Get Model and Tokenizer

In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float8_e5m2,
    bnb_4bit_use_double_quant=True,
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    trust_remote_code=True,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [8]:
if tokenizer.chat_template is not None:
    print(tokenizer.chat_template)
else:
    print("No chat template")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

In [9]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

### Tokenize Datasets

In [10]:
def tokenize_message(example):
    return tokenizer.apply_chat_template(example, tokenize=False)

training_dataset = Dataset.from_pandas(training_dataframe)

### Accuracy Compute

In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    return {}

### LoRA Config

In [12]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [13]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

### TRAINING PARAMETERS

In [14]:
BATCH_SIZE = 2
LEARNING_RATE = 2e-5
LOGGING_STEPS = 200
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 1875
WARMUP_RATIO = 0.1

In [15]:
training_args = SFTConfig(
    output_dir=MODEL_OUTPUT_PATH.resolve(),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    logging_dir=str(LOGS_PATH.resolve()),
    logging_steps=LOGGING_STEPS,
    max_steps=MAX_STEPS,
    fp16=True,
    learning_rate=LEARNING_RATE,
    save_total_limit=1,
    gradient_checkpointing=True,
    push_to_hub=False,
    eval_accumulation_steps=1,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    args=training_args,
    peft_config=peft_config,
)

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\peft\tuners\lora\bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tokenizing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
200,1.574115
400,0.373845
600,0.272614
800,0.228912
1000,0.201556
1200,0.185159
1400,0.177035
1600,0.167994
1800,0.170052


C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences

TrainOutput(global_step=1875, training_loss=0.3639254109700521, metrics={'train_runtime': 9623.4511, 'train_samples_per_second': 1.559, 'train_steps_per_second': 0.195, 'total_flos': 5.506022307785933e+16, 'train_loss': 0.3639254109700521, 'entropy': 0.22170267055432002, 'num_tokens': 8598494.0, 'mean_token_accuracy': 0.963886624375979, 'epoch': 1.0})

In [18]:
save_file_path = Path("~/ft/meta-llama-3_2-1B-instruct").expanduser()
trainer.model.save_pretrained(save_directory=save_file_path.resolve())